In [ ]:
%pip install -q langchain langchain-aws langgraph langsmith boto3 python-dotenv

# Week 6 Wednesday -- Multi-Agent Systems & Deployment

## Overview

Yesterday we explored **context engineering** -- the art of controlling what information flows into and out of LLM calls. Today we scale that concept up: instead of engineering context for a single agent, we'll orchestrate **multiple specialized agents** that each bring focused expertise to complex tasks.

Multi-agent systems are the natural next step when a single agent's tool set or knowledge domain becomes too broad. By dividing responsibilities among specialists -- and carefully engineering the context each one receives -- we can build systems that are more reliable, easier to debug, and simpler to maintain.

## Learning Objectives

By the end of this session, you will be able to:

1. **Explain** when multi-agent architecture is appropriate vs. a single agent
2. **Implement** the supervisor pattern using `create_agent` and tool wrapping
3. **Build** a StateGraph-based supervisor with explicit routing and state
4. **Engineer context** for sub-agents using `ToolRuntime` and `Command`
5. **Deploy** agents locally with `langgraph dev` and understand production deployment to LangSmith

## Session Roadmap

| Stage | Topic | Demo | Key Concept |
|-------|-------|------|-------------|
| 1 | Multi-Agent Systems Overview & Supervisor Pattern | demo_01 | Subagents as tools, supervisor coordination |
| 2 | Sub-agent Context Engineering | demo_02 | StateGraph, ToolRuntime, Command, explicit routing |
| 3 | Deploying Agents to LangSmith | demo_03 | langgraph.json, `langgraph dev`, API endpoints |

---

# Stage 1: Multi-Agent Systems Overview & Supervisor Pattern

## When to Go Multi-Agent

Single agents work well for focused tasks, but real-world applications often span multiple specialized domains. Consider a customer service system handling billing, technical support, and sales -- or an enterprise assistant coordinating calendar, email, and project management. Each domain benefits from a dedicated specialist with its own prompt, tools, and model configuration.

| Use Single Agent When | Use Multi-Agent When |
|----------------------|----------------------|
| Task is focused on one domain | Task spans multiple specialized domains |
| Simple tool set (< 10 tools) | Many tools benefit from logical grouping |
| One persona/expertise fits all | Different personas improve different tasks |
| Straightforward decision making | Complex routing based on task type |

## The Four Multi-Agent Patterns

LangChain supports four main multi-agent patterns:

### 1. Subagents Pattern (Tool Calling) -- Recommended for Most Cases

A supervisor agent calls sub-agents as tools. The supervisor orchestrates via tool calls, each sub-agent is wrapped as a tool, and control flow is simple and explicit.

```
         ┌─────────────────────┐
         │     Supervisor      │
         └──────────┬──────────┘
                    │ (tool calls)
             ┌──────┴──────┐
             ▼             ▼
         ┌───────┐    ┌───────┐
         │ Agent │    │ Agent │
         │   A   │    │   B   │
         └───────┘    └───────┘
```

- Supervisor orchestrates via tool calls
- Each sub-agent is wrapped as a tool
- Simple, explicit control flow
- **Use when:** You need coordinated sub-tasks

### 2. Handoffs Pattern

Agents explicitly pass control to each other. Agent A decides to "hand off" to Agent B, control transfers completely, and the user continues with the new agent. Best for sequential specialization like triage-to-specialist flows.

### 3. Skills Pattern

Multiple agents with overlapping capabilities collaborate on a single result. Results are aggregated from parallel work. Best for tasks where diverse perspectives improve output.

### 4. Router Pattern

A router classifies the request type and sends it to exactly one specialist. Only one agent handles each request. Best for mutually exclusive request types.

### Comparing Patterns

| Pattern | Control | Collaboration | Complexity | Best For |
|---------|---------|---------------|------------|----------|
| Subagents | Centralized | Supervisor-directed | Medium | Most cases |
| Handoffs | Sequential | Explicit transfer | Low | Triage flows |
| Skills | Distributed | Parallel work | High | Creative tasks |
| Router | Initial routing | None | Low | Classification-based |

## The Supervisor Pattern in Detail

Since the supervisor (subagents) pattern covers **90% of multi-agent needs**, this is where we'll focus. The pattern provides:

- **Clear orchestration**: The supervisor controls the workflow
- **Explicit routing**: Tool calls make decisions visible in traces
- **Easy debugging**: LangSmith traces show all agent interactions
- **Flexible composition**: Add or remove sub-agents by adding/removing tools

The flow works like this:

1. User sends a request to the **supervisor**
2. Supervisor analyzes the request and decides which specialist(s) to call
3. Supervisor invokes sub-agents via **tool calls**
4. Each sub-agent processes its task and returns results
5. Supervisor **synthesizes** results into a final response

```
User: "Research AI trends and write a summary"
                    │
                    ▼
         ┌──────────────────┐
         │    Supervisor    │
         │                  │
         │ 1. Understand    │
         │ 2. Plan steps    │
         │ 3. Call tools    │
         └────────┬─────────┘
                  │
    ┌─────────────┼─────────────┐
    ▼             ▼             ▼
┌───────┐    ┌───────┐    ┌───────┐
│Research│    │Analyze│    │ Write │
│ Tool  │    │ Tool  │    │ Tool  │
└───┬───┘    └───┬───┘    └───┬───┘
    │             │             │
    └─────────────┴─────────────┘
                  │
                  ▼
         ┌──────────────────┐
         │    Supervisor    │
         │                  │
         │ Synthesize and   │
         │ respond to user  │
         └──────────────────┘
```

## Wrapping Agents as Tools

The key mechanic that makes the supervisor pattern work is **wrapping sub-agents as tools**. Each sub-agent is created with `create_agent`, then exposed to the supervisor through a `@tool`-decorated function. The tool's **description** is critical -- the supervisor reads it to decide when and how to route requests.

**Key principles for tool wrappers:**

- **Be specific** in descriptions -- tell the supervisor exactly when to use each tool
- **Include negative guidance** -- specify what the tool should NOT be used for
- **Sub-agents must include all results** in their final response -- the supervisor only sees the last message

```python
# The wrapping pattern
from langchain.agents import create_agent
from langchain_core.tools import tool

sub_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[...],
    system_prompt="...",
    name="sub_agent"
)

@tool
def sub_agent_tool(input_param: str) -> str:
    """Description that helps supervisor decide when to use this."""
    result = sub_agent.invoke({
        "messages": [{"role": "user", "content": input_param}]
    })
    return result["messages"][-1].content
```

Let's build a complete supervisor system with three specialists: calendar, email, and research.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain_core.tools import tool

MODEL = "bedrock:us.amazon.nova-2-lite-v1:0"
print(f"Using model: {MODEL}")

### PART 1: Creating Specialized Sub-Agents

Each sub-agent is a specialist with its own system prompt and (optionally) its own tools. Notice the **CRITICAL** instruction in each prompt: the sub-agent must include all relevant information in its final response, because the supervisor only sees that last message -- not the sub-agent's intermediate reasoning.

In [ ]:
calendar_agent = create_agent(
    name="calendar_agent",
    model=MODEL,
    tools=[],
    system_prompt="""You are a calendar management specialist.

When asked to schedule, modify, or check calendar events:
1. Parse the request for date, time, and description
2. Confirm the details
3. Provide clear confirmation

CRITICAL: Include ALL scheduling details in your final response.
The supervisor ONLY sees your final message, not your reasoning."""
)

email_agent = create_agent(
    name="email_agent",
    model=MODEL,
    tools=[],
    system_prompt="""You are an email composition specialist.

When asked to draft or send emails:
1. Clarify recipient, subject, and key points
2. Draft professional, clear emails
3. Provide the complete draft

CRITICAL: Include the FULL email draft in your final response.
The supervisor ONLY sees your final message."""
)

research_agent = create_agent(
    name="research_agent",
    model=MODEL,
    tools=[],
    system_prompt="""You are a research specialist.

When asked to research a topic:
1. Break down the research question
2. Provide key findings and insights
3. Cite sources when applicable

CRITICAL: Include ALL findings in your final response.
The supervisor ONLY sees your final message."""
)

print("Created 3 sub-agents: calendar_agent, email_agent, research_agent")

### PART 2: Wrapping Sub-Agents as Tools

Now we convert each sub-agent into a callable tool. The supervisor will use these tool descriptions to decide which specialist to invoke for any given request. Notice how each tool:

1. Accepts a `request` string parameter
2. Invokes the sub-agent with that request as a user message
3. Returns the sub-agent's **final message content** as the tool result

In [ ]:
@tool
def schedule_meeting(request: str) -> str:
    """Schedule or manage calendar events.

    Use for: meetings, appointments, reminders, calendar queries.
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


@tool
def compose_email(request: str) -> str:
    """Draft or send emails.

    Use for: writing emails, reviewing drafts, email-related tasks.
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


@tool
def research_topic(request: str) -> str:
    """Research a topic and gather information.

    Use for: fact-finding, background research, answering questions.
    """
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


print("Wrapped sub-agents as tools: schedule_meeting, compose_email, research_topic")

### PART 3: Creating the Supervisor Agent

The supervisor is itself an agent created with `create_agent`. It receives the three tool-wrapped sub-agents and a system prompt that describes its role as coordinator. The supervisor doesn't do the specialized work -- it **delegates** to the right specialist and **synthesizes** results.

In [ ]:
supervisor = create_agent(
    name="supervisor_demo",
    model=MODEL,
    tools=[schedule_meeting, compose_email, research_topic],
    system_prompt="""You are a personal assistant supervisor.

You coordinate specialized agents:
- schedule_meeting: For calendar and scheduling tasks
- compose_email: For drafting and sending emails
- research_topic: For researching and gathering information

For each user request:
1. Determine which specialist(s) can help
2. Delegate to the appropriate agent(s)
3. Synthesize results into a coherent response

You can use multiple specialists for complex requests."""
)

print("Supervisor agent created with 3 tool-wrapped sub-agents")

### PART 4: Testing the Supervisor

Let's send a couple of requests to see how the supervisor routes them to the appropriate sub-agents. Watch how the supervisor decides which tool to call based on the request content.

In [ ]:
requests = [
    "Schedule a meeting with the team for next Tuesday at 2pm to discuss Q4 goals",
    "Research the latest trends in AI agents",
]

for request in requests:
    print(f"\n{'='*60}")
    print(f"Request: {request}")
    print('='*60)
    result = supervisor.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    print(f"\nResponse:\n{result['messages'][-1].content}")

---

# Stage 2: Sub-agent Context Engineering with StateGraph

## From Tool Calling to Explicit Routing

In Stage 1 we used the **tool-calling** approach: sub-agents are wrapped as tools, and the supervisor's LLM decides which tool to call. This is simple and works great for most cases.

But sometimes you need more control:

- **Custom state** shared across agents
- **Explicit routing logic** you can inspect and test
- **Fine-grained control** over what information flows between agents
- **Conditional edges** with deterministic branching

LangGraph's `StateGraph` gives you this control. Instead of wrapping agents as tools, you define them as **graph nodes** connected by **edges** with **conditional routing**.

## Context Engineering for Sub-Agents

In the basic supervisor pattern, sub-agents receive only what's passed as tool arguments. But production systems often need:

- **Conversation history** passed to sub-agents for context
- **User preferences** from state available during execution
- **Structured results** that update supervisor state, not just text

LangChain provides three mechanisms for this:

### ToolRuntime -- Controlling Input

Access the full runtime context (state, store, config) inside tool wrappers:

```python
from langchain.tools import tool, ToolRuntime

@tool
def research_with_context(query: str, runtime: ToolRuntime) -> str:
    messages = runtime.state.get("messages", [])
    preferences = runtime.state.get("user_preferences", {})
    # Build context-aware prompt using history and preferences
```

### Command -- Controlling Output

Return structured results that update supervisor state:

```python
from langgraph.types import Command

@tool
def research_with_state_update(query: str, runtime: ToolRuntime) -> Command:
    result = research_agent.invoke(...)
    return Command(
        result=findings,             # What supervisor sees as tool result
        update={"key_facts": [...]}  # State updates
    )
```

### InjectedToolCallId -- Tracking Executions

Track which tool call produced which result for audit and debugging.

Now let's build the same supervisor system using `StateGraph` for explicit control flow.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage


class SupervisorState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_agent: Literal["calendar", "email", "research", "respond", "__end__"]
    task_result: str


print("State schema defined: messages, next_agent, task_result")

### PART 1: Supervisor Node with Routing Logic

The supervisor node does two things depending on the current state:

1. **If there's a task result**: Synthesize the specialist's response into a final answer
2. **If there's a new request**: Route to the appropriate specialist by analyzing the message content

This explicit routing replaces the implicit tool-calling decision of Stage 1.

In [ ]:
def supervisor_node(state: SupervisorState) -> dict:
    model = init_chat_model(MODEL)

    last_message = state["messages"][-1].content if state["messages"] else ""

    if state.get("task_result"):
        response = model.invoke([
            {"role": "system", "content": """You are a personal assistant supervisor.
Synthesize the specialist's response into a helpful final answer for the user."""},
            {"role": "user", "content": f"Original request: {last_message}\n\nSpecialist result: {state['task_result']}"}
        ])
        return {
            "messages": [AIMessage(content=response.content)],
            "next_agent": "__end__",
            "task_result": ""
        }

    routing_response = model.invoke([
        {"role": "system", "content": """You are a supervisor that routes requests to specialists.

Available specialists:
- calendar: Schedule meetings, appointments, reminders
- email: Draft or send emails
- research: Research topics, gather information

Respond with ONLY the specialist name (calendar, email, or research).
If the request doesn't fit any specialist, respond with 'respond' to handle directly."""},
        {"role": "user", "content": last_message}
    ])

    route = routing_response.content.strip().lower()
    if route not in ["calendar", "email", "research"]:
        route = "respond"

    return {"next_agent": route}


print("Supervisor node defined with routing logic")

### PART 2: Sub-Agent Nodes

Each specialist is a graph node that receives the full state, processes the request using its own model call with a specialized system prompt, and returns the result to `task_result`. The supervisor will then pick this up and synthesize it into a final response.

In [ ]:
def calendar_node(state: SupervisorState) -> dict:
    model = init_chat_model(MODEL)
    last_message = state["messages"][-1].content

    response = model.invoke([
        {"role": "system", "content": """You are a calendar management specialist.

When asked to schedule, modify, or check calendar events:
1. Parse the request for date, time, and description
2. Confirm the details
3. Provide clear confirmation

CRITICAL: Include ALL scheduling details in your response."""},
        {"role": "user", "content": last_message}
    ])

    return {"task_result": response.content, "next_agent": "supervisor"}


def email_node(state: SupervisorState) -> dict:
    model = init_chat_model(MODEL)
    last_message = state["messages"][-1].content

    response = model.invoke([
        {"role": "system", "content": """You are an email composition specialist.

When asked to draft or send emails:
1. Clarify recipient, subject, and key points
2. Draft professional, clear emails
3. Provide the complete draft

CRITICAL: Include the FULL email draft in your response."""},
        {"role": "user", "content": last_message}
    ])

    return {"task_result": response.content, "next_agent": "supervisor"}


def research_node(state: SupervisorState) -> dict:
    model = init_chat_model(MODEL)
    last_message = state["messages"][-1].content

    response = model.invoke([
        {"role": "system", "content": """You are a research specialist.

When asked to research a topic:
1. Break down the research question
2. Provide key findings and insights
3. Cite sources when applicable

CRITICAL: Include ALL findings in your response."""},
        {"role": "user", "content": last_message}
    ])

    return {"task_result": response.content, "next_agent": "supervisor"}


def respond_node(state: SupervisorState) -> dict:
    model = init_chat_model(MODEL)
    last_message = state["messages"][-1].content

    response = model.invoke([
        {"role": "system", "content": "You are a helpful personal assistant. Respond directly to the user."},
        {"role": "user", "content": last_message}
    ])

    return {
        "messages": [AIMessage(content=response.content)],
        "next_agent": "__end__"
    }


print("Sub-agent nodes defined: calendar, email, research, respond")

### PART 3: Building and Compiling the Graph

Now we wire everything together using `StateGraph`. The key difference from the tool-calling approach:

- **Nodes** are agent functions, not tools
- **Conditional edges** from the supervisor handle routing
- **Regular edges** from sub-agents return control to the supervisor
- The graph is **compiled** into a runnable agent

This gives us full visibility into the control flow -- we can inspect edges, test routing logic, and visualize the graph.

In [ ]:
def route_from_supervisor(state: SupervisorState) -> str:
    return state["next_agent"]


workflow = StateGraph(SupervisorState)

workflow.add_node("supervisor", supervisor_node)
workflow.add_node("calendar", calendar_node)
workflow.add_node("email", email_node)
workflow.add_node("research", research_node)
workflow.add_node("respond", respond_node)

workflow.add_edge(START, "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "calendar": "calendar",
        "email": "email",
        "research": "research",
        "respond": "respond",
        "__end__": END
    }
)

workflow.add_edge("calendar", "supervisor")
workflow.add_edge("email", "supervisor")
workflow.add_edge("research", "supervisor")
workflow.add_edge("respond", END)

graph_agent = workflow.compile()

print("StateGraph compiled successfully")
print("Nodes: supervisor, calendar, email, research, respond")
print("Flow: START -> supervisor -> [calendar|email|research|respond] -> supervisor -> END")

### PART 4: Testing the StateGraph Supervisor

Let's run the same requests through the StateGraph-based supervisor. Compare the results with Stage 1's tool-calling approach -- the outputs should be similar, but the internal execution path is explicit rather than implicit.

In [ ]:
test_requests = [
    "Schedule a meeting with the team for next Tuesday at 2pm to discuss Q4 goals",
    "Research the latest trends in AI agents",
]

for request in test_requests:
    print(f"\n{'='*60}")
    print(f"Request: {request}")
    print('='*60)
    result = graph_agent.invoke({
        "messages": [HumanMessage(content=request)],
        "next_agent": "",
        "task_result": ""
    })
    print(f"\nResponse:\n{result['messages'][-1].content}")

## Testing Multi-Agent Systems

Multi-agent systems are complex. Bugs can hide at the individual agent level, in agent-to-agent communication, in supervisor routing decisions, and in state management across agents. The testing pyramid applies:

**Unit Tests** (many): Test each sub-agent in isolation. Does the research agent return relevant findings? Does the email agent format properly?

```python
def test_research_agent_returns_findings():
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": "Research Python trends"}]
    })
    response = result["messages"][-1].content
    assert len(response) > 100
    assert "python" in response.lower()
```

**Integration Tests** (some): Test agent interactions. Does the supervisor correctly route research requests? Can it chain multiple agents?

```python
def test_supervisor_delegates_to_research():
    result = supervisor.invoke({
        "messages": [{"role": "user", "content": "Find information about quantum computing"}]
    })
    response = result["messages"][-1].content
    assert "quantum" in response.lower()
```

**End-to-End Tests** (few): Test the full workflow with complex multi-step requests.

### Debugging with LangSmith

Enable tracing to see the full execution flow:

```python
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "multi-agent-tests"
```

Traces show: supervisor receives request -> decides which tool/node -> sub-agent executes -> result returns -> supervisor synthesizes response. This visibility is invaluable for debugging routing issues and context loss between agents.

---

# Stage 3: Deploying Agents to LangSmith

## From Development to Production

Development is only half the journey. Production deployment requires reliable hosting, version control, a testing environment, and API access for applications. LangSmith provides infrastructure for deploying, testing, and serving agents.

> **Important:** LangSmith Cloud deployments are a **paid service**. For this demo, local testing with `langgraph dev` is sufficient. The cloud deployment steps are provided as a reference for production scenarios.

## Deployment Flow

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Local Dev     │ ──► │   GitHub Repo   │ ──► │   LangSmith     │
│                 │     │                 │     │   Deployment    │
│ - agent.py      │     │ - langgraph.json│     │ - Hosted        │
│ - requirements  │     │ - agent code    │     │ - API endpoint  │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

## Step 1: Project Structure

Your agent project needs these files:

```
my-agent/
├── agent.py           # Agent definition (exports `agent` variable)
├── requirements.txt   # Dependencies
├── langgraph.json     # LangGraph configuration
└── .env.example       # Environment variables template
```

The critical file is `langgraph.json`, which tells LangSmith where to find your agent's compiled graph.

## Step 2: Local Testing with `langgraph dev`

Before deploying, always test locally:

```bash
cd your-agent-directory
langgraph dev
```

This starts a local server at `http://127.0.0.1:2024`. Open LangSmith Studio at `https://smith.langchain.com/studio/` to interact with your agent visually, inspect traces, and debug state.

## Step 3: Deploy to LangSmith Cloud (Optional)

1. Push your code to GitHub
2. Go to LangSmith > Deployments > New Deployment
3. Connect your repository
4. Configure environment variables
5. Deploy

## Comparison: create_agent vs StateGraph for Deployment

Both deploy identically! The choice depends on your needs:

| Approach | Pros |
|----------|------|
| `create_agent` (Demo 01) | Simpler code, less boilerplate, automatic tool execution |
| `StateGraph` (Demo 02) | Explicit control flow, custom state, fine-grained routing |

In [ ]:
import json

langgraph_config_tool_calling = {
    "dependencies": ["."],
    "graphs": {
        "supervisor_langchain": "./demo_01_supervisor_agent.py:agent"
    },
    "env": ".env"
}

langgraph_config_stategraph = {
    "dependencies": ["."],
    "graphs": {
        "supervisor_langgraph": "./demo_02_subagent_context.py:agent"
    },
    "env": ".env"
}

print("langgraph.json for tool-calling approach (Demo 01):")
print(json.dumps(langgraph_config_tool_calling, indent=2))
print("\nlanggraph.json for StateGraph approach (Demo 02):")
print(json.dumps(langgraph_config_stategraph, indent=2))

In [ ]:
print("""=== Calling a Deployed Agent via the LangGraph SDK ===

from langgraph_sdk import get_client

client = get_client(url="https://your-deployment.langsmith.com")

thread = await client.threads.create()

result = await client.runs.create(
    thread_id=thread["thread_id"],
    assistant_id="supervisor_langchain",
    input={
        "messages": [{"role": "user", "content": "Schedule a meeting for Tuesday at 2pm"}]
    }
)

=== Production Best Practices ===

1. CHECKPOINTING
   - Dev: InMemorySaver (local testing only)
   - Prod: LangSmith handles this automatically

2. ENVIRONMENT VARIABLES
   - Never commit API keys
   - Use LangSmith Secrets for production

3. MONITORING
   - All runs are traced in LangSmith automatically
   - Set up alerts for errors and latency

4. VERSIONING
   - Use git branches for different versions
   - LangSmith supports multiple deployment versions
   - Roll back easily if issues arise

5. SCALING
   - LangSmith auto-scales based on demand
   - Configure concurrency limits if needed
""")

---

# Key Takeaways

1. **Multi-agent systems** divide complex tasks among specialized agents, each optimized for its domain -- use them when a single agent's tool set or persona would be too broad
2. **The supervisor pattern** covers 90% of multi-agent needs: a coordinator calls sub-agents as tools (or graph nodes), delegates tasks, and synthesizes results
3. **Wrapping agents as tools** with `@tool` is the key mechanic -- tool descriptions are routing instructions that guide the supervisor's decisions
4. **Context engineering applies at the multi-agent level** too: use `ToolRuntime` to pass state into sub-agents, and `Command` to return structured results that update state
5. **Two implementation approaches** serve different needs: `create_agent` with tool wrapping for simplicity, or `StateGraph` for explicit control flow and custom state
6. **LangSmith deployment** follows a simple flow: structure your project with `langgraph.json`, test locally with `langgraph dev`, and optionally deploy to the cloud for production

## Up Next: Thursday -- Human-in-the-Loop & Guardrails

Tomorrow we add the final critical layer: **human oversight**. We'll explore:

- **Human-in-the-loop patterns** -- interrupt agents for approval before executing sensitive actions
- **Guardrails** -- input validation, output filtering, and safety boundaries
- Combining human oversight with the multi-agent patterns we built today